# Day 1.7 — Project — Smart Research Assistant
Everything is on the table: `chat`, a validated contract (`ResearchSummary`), two safe tools,
and the bounded loop. The project wires them together and then judges the assistant by its
**evidence and trace**, not by how fluent the final paragraph sounds.

### Step 1 — Assemble it

The assistant is `run_agent` with a strict final format. The answer is parsed into
`ResearchSummary`, so downstream code receives typed data or a clear failure.

In [ ]:
ASSISTANT_PROMPT = (SYSTEM_PROMPT + " Use the calculator for arithmetic and search_local_notes for course concepts. "
                    "When you have the evidence, answer with the requested JSON only.")

def research(question, model=chat, max_steps=5):
    """Run the assistant and return a typed result plus its trace."""
    run = run_agent(question, TOOLS, model=model, max_steps=max_steps, system=ASSISTANT_PROMPT,
                    response_format=RESEARCH_FORMAT)
    summary, reason = (validate_summary(run["answer"]) if run["status"] == "completed" else (None, run["error"]))
    return {"question": question, "status": run["status"] if summary else "failed",
            "summary": summary, "reason": reason, "tools_used": run["tools_used"],
            "steps": run["steps"], "usage": run["usage"], "messages": run["messages"]}

outcome = research("What is structured output, and what is 144 / 12?")
print("\nstatus     :", outcome["status"], "|", outcome["reason"])
print("tools used :", outcome["tools_used"], "| steps:", outcome["steps"], "| tokens:", outcome["usage"]["total_tokens"])
if outcome["summary"]:
    print("summary    :", outcome["summary"].summary[:200])
    print("confidence :", outcome["summary"].confidence)

### Step 2 — A behaviour suite

Three questions with known expectations: which tools *should* be used, did the run finish, and
was the final answer valid. This is a tiny evaluation; Day 2 grows it into a golden set.

In [ ]:
SUITE = [
    ("What is 37 * 19?",                         {"calculator"}),
    ("What is an agent?",                        {"search_local_notes"}),
    ("What is a tool, and what is 12 * 9?",      {"calculator", "search_local_notes"}),
]

print(f"{'question':<42} {'status':<10} {'tools':<34} valid")
for question, expected_tools in SUITE:
    outcome = research(question)
    tools_ok = expected_tools.issubset(set(outcome["tools_used"]))
    valid = outcome["summary"] is not None
    verdict = "PASS" if (outcome["status"] == "completed" and tools_ok and valid) else "CHECK"
    print(f"{question:<42} {outcome['status']:<10} {str(sorted(set(outcome['tools_used']))):<34} {valid}  {verdict}")

### Step 3 — Three failures, each caught by a different layer

Do not call every failure a "hallucination". Locate the layer that failed: the tool layer, the
loop, or the output contract. Each one below is stopped by the code you wrote today.

In [ ]:
# Failure 1: an invalid tool argument (tool layer). We call the executor directly.
print("1. tool layer   :", execute_tool_call({"id": "f1", "name": "calculator", "arguments": {"expression": "10 / 0"}}, TOOLS))

# Failure 2: a model that repeats itself (loop layer).
repeated = research("What is 1 + 1?", model=stuck)
print("2. loop layer   :", repeated["status"], "|", repeated["reason"])

# Failure 3: a final answer that breaks the contract (output layer).
broken_final = make_fake_model([_mock_reply(json.dumps({"topic": "x", "summary": "y", "key_points": [], "confidence": 2}))])
invalid = research("Anything", model=broken_final)
print("3. output layer :", invalid["status"], "|", invalid["reason"])

### Try it yourself

Add a third tool: `word_count(text)` that returns the number of words. Register it with
`make_tool` and ask the assistant "How many words are in 'agents use tools through code'?".
Predict which tool it should use before running.

In [ ]:
# --- Worked solution ---------------------------------------------------------------
class WordCountArgs(BaseModel):
    text: str = Field(min_length=1, max_length=500)

def word_count(text: str) -> str:
    return str(len(text.split()))                       # the whole tool: split on whitespace, count

TOOLS["word_count"] = make_tool("word_count", "Count the words in a piece of text.", word_count, WordCountArgs)

# The mock model does not know this tool, so we script the request to test OUR side of the contract;
# in LIVE mode the real model chooses the tool itself.
scripted = make_fake_model([
    _mock_reply(calls=[{"id": "w1", "name": "word_count", "arguments": {"text": "agents use tools through code"}}]),
    _mock_reply(json.dumps({"topic": "word count", "summary": "The sentence has 5 words.", "key_points": ["5 words"],
                            "tools_used": ["word_count"], "confidence": 0.95})),
])
outcome = research("How many words are in 'agents use tools through code'?", model=chat if LIVE else scripted)
print("\nstatus:", outcome["status"], "| tools used:", outcome["tools_used"], "| answer:", outcome["summary"].summary if outcome["summary"] else outcome["reason"])

### Checkpoint

**1. The assistant produced a fluent paragraph with `confidence: 0.9`. What would you check before trusting it?**

<details><summary>Show answer</summary>

The trace: which tools ran, with what arguments, and what they returned. Then whether the summary is consistent with those observations. Fluency and a confidence number are model output, not evidence.

</details>

**2. Which parts of today's assistant are *application-specific* and which would you reuse on Day 5?**

<details><summary>Show answer</summary>

Reusable: `chat`, `make_tool`/`execute_tool_call`, the bounded loop, validation helpers. Application-specific: the prompt, the two tools, the `ResearchSummary` contract, and the suite. Day 5 turns the reusable parts into a harness.

</details>

### Recap

- **Limitation seen:** a fluent answer says nothing about whether the process was sound.
- **Layer added:** a typed final contract, a behaviour suite, and failure cases mapped to layers.
- **Evidence:** the suite table, and three failures each stopped with a named reason.